[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C14_DL_Theory_Data_Course/02_grokking_emergence/02_grokking_emergence.ipynb)

# 02 · Grokking 与涌现（用 numpy 可靠复现）

目标：把 **grokking（记忆→泛化的延迟迁移，weight decay 驱动）** 与 **涌现的「度量假象」（Schaeffer）** 用纯 numpy 可靠复现, 用 `assert` 钉死。

路线：构造记忆解(零空间分量) → GD+weight decay 让测试误差延迟暴跌(grokking) → WD 开/关对照 → progress measure(cos 对齐) → 涌现: 同一平滑能力 exact-match vs 连续度量 → ✏️ 练习 → 📖 答案 → 🧪 真实数据胶囊(模算术结构 + 缩放律平滑性)。

> 心智模型：**grokking = 在「训练误差=0」的流形上, weight decay 把高范数记忆解缓慢挪向低范数泛化解**；**涌现「跳变」常是把平滑能力喂给不连续度量的产物**。

## 1 · 构造一个「记忆解」

过参数化回归 `d=60 > n=55`。真目标由低范数 `w_gen` 给出 `y=X@w_gen`。

**记忆解** = `w_gen + 训练特征零空间方向`。沿零空间 `v`(满足 `X_tr@v=0`)加任何量都**不改训练预测**(完美拟合训练集), 但在测试集上是纯噪声(不泛化)。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
n, d = 55, 60                          # 轻度过参数化: d>n -> 记忆可能; n 够大 -> 泛化解可恢复
X_tr = rng.standard_normal((n, d))
X_te = rng.standard_normal((400, d))
w_gen = rng.standard_normal(d); w_gen /= np.linalg.norm(w_gen)   # 低范数泛化解(真相)
y_tr = X_tr @ w_gen
y_te = X_te @ w_gen

# 训练特征零空间(维度 d-n=5): 沿这些方向不改训练预测
U, S, Vt = np.linalg.svd(X_tr)
null_space = Vt[n:]                    # (d-n, d)
assert null_space.shape[0] == d - n
assert np.allclose(X_tr @ null_space[0], 0, atol=1e-8), '零空间方向: X_tr@v=0'

w_memo = w_gen + 6.0 * null_space[0] + 3.6 * null_space[1]       # 一个记忆解
def tr_mse(w): return np.mean((X_tr @ w - y_tr) ** 2)
def te_mse(w): return np.mean((X_te @ w - y_te) ** 2)
print(f'记忆解: 训练MSE={tr_mse(w_memo):.2e}  测试MSE={te_mse(w_memo):.2f}  |w|={np.linalg.norm(w_memo):.2f}')
print(f'泛化解: 训练MSE={tr_mse(w_gen):.2e}  测试MSE={te_mse(w_gen):.2f}  |w|={np.linalg.norm(w_gen):.2f}')
assert tr_mse(w_memo) < 1e-6, '记忆解完美拟合训练集'
assert te_mse(w_memo) > 10 * te_mse(w_gen), '记忆解测试差; 泛化解测试好'
assert np.linalg.norm(w_memo) > 3 * np.linalg.norm(w_gen), '记忆解范数远大于泛化解'
print('✅ 记忆解: 训练满分但测试差、范数大; 泛化解: 两者都好、范数小')

## 2 · Grokking：weight decay 让测试误差延迟暴跌

从记忆解 `w_memo` 出发, 跑「GD(拟合训练) + weight decay」。
训练误差全程≈0(记忆从第0步就完成); **weight decay 把零空间(记忆)分量按 `(1-ηλ)^t` 几何衰减掉**, 解滑向 `w_gen` -> 测试误差延迟暴跌。这就是 grokking。

In [ ]:
def train_grok(wd, steps=6000, lr=0.05, w0=None, record_every=500):
    w = (w_memo if w0 is None else w0).copy()
    hist = []
    for s in range(steps + 1):
        grad = 2.0 * X_tr.T @ (X_tr @ w - y_tr) / n + wd * w   # 训练梯度 + weight decay
        w = w - lr * grad
        if s % record_every == 0:
            hist.append((s, tr_mse(w), te_mse(w), float(np.linalg.norm(w))))
    return w, hist

w_final, hist = train_grok(wd=0.05, steps=6000)
print(f"{'step':>6}{'train_MSE':>12}{'test_MSE':>12}{'|w|':>8}")
for s, tr, te, nw in hist:
    print(f'{s:>6}{tr:>12.4f}{te:>12.3f}{nw:>8.2f}')

te_start = hist[0][2]; te_end = hist[-1][2]
print(f'\n测试MSE: {te_start:.1f} -> {te_end:.3f}  (延迟暴跌 {te_start/te_end:.0f}x)')
assert all(h[1] < 1e-2 for h in hist), '训练误差应全程≈0(记忆早已完成)'
assert te_end < 0.3, 'grokking 后测试误差应很小(泛化了)'
assert te_start > 30 * te_end, '测试误差应延迟大幅下降'
assert hist[-1][3] < hist[0][3] / 3, 'weight decay 应大幅压低范数'
print('✅ GROKKING 复现: 训练全程满分, 测试误差延迟暴跌, 范数被 weight decay 压低')

## 3 · 对照实验：没有 weight decay 就不 grok

**grokking 的引擎是 weight decay。** 关掉它(`wd=0`): 训练误差从第0步就是0, 梯度消失, 没有任何力推动解离开记忆解 -> 测试误差**冻结**, 永不泛化。

In [ ]:
_, hist_on  = train_grok(wd=0.05, steps=6000)
_, hist_off = train_grok(wd=0.0,  steps=6000)

print(f"{'step':>6}{'test_MSE(WD开)':>16}{'test_MSE(WD关)':>16}")
for (s, _, te_on, _), (_, _, te_off, _) in zip(hist_on, hist_off):
    print(f'{s:>6}{te_on:>16.3f}{te_off:>16.3f}')

assert hist_on[-1][2] < 0.3, 'WD开: 最终泛化(测试误差小)'
assert hist_off[-1][2] > 30, 'WD关: 测试误差冻结在记忆解(不泛化)'
assert abs(hist_off[0][2] - hist_off[-1][2]) < 1.0, 'WD关: 测试误差几乎不动'
assert hist_off[-1][3] > 5.0, 'WD关: 范数不下降(仍是高范数记忆解)'
print('\n✅ 唯一区别是 weight decay: 开则 grok(泛化), 关则永远停在记忆解 —— WD 是 grokking 的引擎')

## 4 · Progress measure：顿悟前就平滑爬升的内部量

Nanda 2023: grokking 的「突然」其实是某个**平滑进展量累积过阈**。这里的 progress measure = `cos(w, w_gen)`(解的方向与真泛化解的对齐度)。它在测试误差暴跌**之前**就平滑爬升, 预示顿悟在酝酿。

In [ ]:
def cos_sim(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

def train_with_progress(wd=0.05, steps=6000, record_every=500):
    w = w_memo.copy(); rows = []
    for s in range(steps + 1):
        grad = 2.0 * X_tr.T @ (X_tr @ w - y_tr) / n + wd * w
        w = w - lr_default * grad
        if s % record_every == 0:
            rows.append((s, te_mse(w), cos_sim(w, w_gen)))
    return rows

lr_default = 0.05
rows = train_with_progress()
print(f"{'step':>6}{'test_MSE':>12}{'cos(w,w_gen)':>14}")
for s, te, cs in rows:
    print(f'{s:>6}{te:>12.3f}{cs:>14.3f}')

cos_curve = [r[2] for r in rows]
assert cos_curve[0] < 0.2, '初始: 方向与真解几乎正交(记忆解)'
assert cos_curve[-1] > 0.9, '最终: 方向高度对齐真解(泛化了)'
assert all(cos_curve[i+1] >= cos_curve[i] - 1e-6 for i in range(len(cos_curve)-1)), 'progress measure 单调爬升(平滑)'
print('\n✅ cos(w,w_gen) 平滑爬升 0.08->0.95 —— 这是 progress measure: 顿悟不是凭空, 是平滑进展过阈')

## 5 · 涌现的「度量假象」：同一能力，两种曲线

Schaeffer 2023: 许多「涌现」来自**不连续度量**。玩具: 单步正确率 `p_c` 随规模**平滑**上升; 任务要连续 `k` 步**全对**(exact-match): `成功率 = p_c^k`。

**同一条平滑的 `p_c`, exact-match 看是「涌现跳变」, 连续度量看是「平滑增长」。**

In [ ]:
scales = np.linspace(0.0, 1.0, 21)        # 模型规模(归一化)
p_correct = scales ** 1.3                 # 单步正确率: 平滑、连续
k = 6                                     # 任务需连续 k 步全对
exact_match = p_correct ** k              # all-or-nothing 度量(不连续)

print(f"{'规模':>6}{'单步正确率(连续)':>18}{'6步全对(不连续)':>18}")
for s, pc, em_ in list(zip(scales, p_correct, exact_match))[::2]:
    print(f'{s:>6.2f}{pc:>18.3f}{em_:>18.4f}')

mid = 10                                  # 中段(规模=0.5)
assert p_correct[mid] > 0.3, '连续度量: 中段已明显非零(平滑可见进展)'
assert exact_match[mid] < 0.05, '不连续度量: 中段仍贴地(看不到进展)'
assert exact_match[-1] > 0.5, '不连续度量: 末端才骤升(像涌现)'
# 连续度量全程单调平滑(相邻增量稳定); 不连续度量末端增量远大于中段(跳变)
d_cont = np.diff(p_correct); d_em = np.diff(exact_match)
assert d_em[-1] > 3 * d_em[mid], 'exact-match 末端斜率远陡于中段(突变特征)'
print('\n✅ 同一平滑能力: exact-match 制造「涌现」假象, 连续度量显示真实的平滑增长')

## 6 · 连续度量可外插：用前半段预测后半段

Schaeffer 的实用判据: **连续度量平滑到可外插**, 不连续度量不能。
验证: 用 `p_c` 前 60% 的点拟合一条曲线, 看能否预测后段(误差小); 对 exact-match 同样外插则误差大。

In [ ]:
def extrapolate_error(x, y, frac=0.6, deg=2):
    '''用前 frac 的点拟合 deg 次多项式, 预测后段, 返回后段预测的 RMSE。'''
    m = int(len(x) * frac)
    coef = np.polyfit(x[:m], y[:m], deg)
    pred = np.polyval(coef, x[m:])
    return np.sqrt(np.mean((pred - y[m:]) ** 2))

err_cont = extrapolate_error(scales, p_correct, frac=0.6, deg=2)
err_em   = extrapolate_error(scales, exact_match, frac=0.6, deg=2)
print(f'外插后段 RMSE:  连续度量 p_c = {err_cont:.4f}')
print(f'外插后段 RMSE:  exact-match  = {err_em:.4f}')
assert err_cont < err_em, '连续度量更可外插(用前段能预测后段)'
print('✅ 连续度量平滑可外插 -> 能提前预见能力增长; 不连续度量不能 -> 「涌现」难预测(因为是度量假象)')

---
## ✏️ 练习 1：grokking 曲线

实现 `grok_curve(wd, steps, lr)`：从 `w_memo` 出发训练, 返回 `(steps_list, train_mse_list, test_mse_list)`(每 500 步记录)。
用它验证 grokking 三特征: 训练全程≈0、测试延迟下降、范数被压低。

In [ ]:
def grok_curve(wd=0.05, steps=6000, lr=0.05, record_every=500):
    # TODO: 从 w_memo.copy() 出发, 每步 grad = 2*X_tr.T@(X_tr@w-y_tr)/n + wd*w; w -= lr*grad
    #       每 record_every 步记录 (step, tr_mse(w), te_mse(w))
    #       返回三个等长 list: steps, train_mses, test_mses
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
ss, trs, tes = grok_curve(wd=0.05, steps=6000)
assert len(ss) == len(trs) == len(tes)
assert max(trs) < 1e-2, '训练误差应全程≈0'
assert tes[0] > 30 * tes[-1], '测试误差应延迟大幅下降(grokking)'
assert tes[-1] < 0.3, '最终应泛化'
print(f'测试MSE: {tes[0]:.1f} -> {tes[-1]:.3f}')
print('✅ 练习 1 通过: 复现 grokking 曲线(训练早满分、测试后暴跌)')

## ✏️ 练习 2：weight decay 扫描

实现 `wd_sweep(wd_list, steps)`：对每个 weight decay 强度, 返回**最终测试 MSE**。
验证: `wd=0` 不泛化(测试 MSE 大); 适当 `wd>0` 泛化(测试 MSE 小)。

In [ ]:
def wd_sweep(wd_list, steps=6000, lr=0.05):
    # TODO: 对每个 wd, 从 w_memo.copy() 训练 steps 步, 返回最终 te_mse(w) 的 list
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
wds = [0.0, 0.01, 0.05]
final_te = wd_sweep(wds)
for wd, te in zip(wds, final_te):
    print(f'wd={wd}: 最终测试MSE={te:.3f}')
assert final_te[0] > 30, 'wd=0 不泛化(测试MSE大)'
assert final_te[-1] < 0.3, 'wd=0.05 泛化(测试MSE小)'
assert final_te[0] > final_te[-1], 'weight decay 越强(到合理范围)越能泛化'
print('✅ 练习 2 通过: weight decay 是 grokking 的开关')

## ✏️ 练习 3：涌现度量

实现 `emergence_curves(p_correct, k)`：返回 `(exact_match, mean_token_acc)`。
- `exact_match = p_correct ** k`(k 步全对, 不连续);
- `mean_token_acc = p_correct`(平均单步正确率, 连续, 这里就是 p_correct 本身)。
验证 exact-match 中段贴地、末端骤升, 连续度量平滑。

In [ ]:
def emergence_curves(p_correct, k=6):
    # TODO: 返回 (p_correct**k, p_correct)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
pc = np.linspace(0, 1, 21) ** 1.3
em_, tok = emergence_curves(pc, k=6)
mid = 10
assert tok[mid] > 0.3 and em_[mid] < 0.05, '连续度量中段有进展、exact-match 中段贴地'
assert em_[-1] > 0.5, 'exact-match 末端骤升'
assert np.all(np.diff(tok) >= -1e-9), '连续度量单调平滑'
# 「涌现」的尖锐度: exact-match 末段斜率 / 中段斜率
sharp = np.diff(em_)[-1] / (np.diff(em_)[mid] + 1e-9)
assert sharp > 3, 'exact-match 末端比中段陡得多(像跳变)'
print(f'exact-match 末/中段斜率比 = {sharp:.1f}x (跳变越尖此值越大)')
print('✅ 练习 3 通过: 度量的连续性决定曲线是「涌现」还是「平滑」')

## ✏️ 练习 4：平滑指标(把跳变变回平滑)

给定一个看似「涌现」的 exact-match 曲线, 实现 `recover_smooth(exact_match, k)`：
**反推**底层连续能力 `p_correct = exact_match ** (1/k)`, 并验证它平滑、可外插。
这正是 Schaeffer 的操作: 用合适的(去阈值的)度量, 把表观跳变还原成平滑增长。

In [ ]:
def recover_smooth(exact_match, k=6):
    # TODO: 返回 exact_match ** (1.0 / k)  (反推单步正确率)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
scales = np.linspace(0, 1, 21)
pc_true = scales ** 1.3
em_obs = pc_true ** 6                       # 我们「观测」到的涌现曲线
pc_rec = recover_smooth(em_obs, k=6)        # 反推
assert np.allclose(pc_rec, pc_true, atol=1e-9), '应还原出底层平滑能力'
# 还原后可外插(用前60%预测后段, 误差小)
def extrap_rmse(x, y, frac=0.6, deg=2):
    m = int(len(x)*frac); coef = np.polyfit(x[:m], y[:m], deg)
    return np.sqrt(np.mean((np.polyval(coef, x[m:]) - y[m:])**2))
assert extrap_rmse(scales, pc_rec) < extrap_rmse(scales, em_obs), '还原后比原exact-match更可外插'
print('✅ 练习 4 通过: 去阈值的度量把「涌现」还原为可外插的平滑增长(Schaeffer 的核心操作)')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def grok_curve(wd=0.05, steps=6000, lr=0.05, record_every=500):
    w = w_memo.copy(); steps_l, tr_l, te_l = [], [], []
    for s in range(steps + 1):
        grad = 2.0 * X_tr.T @ (X_tr @ w - y_tr) / n + wd * w
        w = w - lr * grad
        if s % record_every == 0:
            steps_l.append(s); tr_l.append(tr_mse(w)); te_l.append(te_mse(w))
    return steps_l, tr_l, te_l

In [ ]:
# 练习 2 参考答案
def wd_sweep(wd_list, steps=6000, lr=0.05):
    out = []
    for wd in wd_list:
        w = w_memo.copy()
        for _ in range(steps + 1):
            grad = 2.0 * X_tr.T @ (X_tr @ w - y_tr) / n + wd * w
            w = w - lr * grad
        out.append(te_mse(w))
    return out

In [ ]:
# 练习 3 参考答案
def emergence_curves(p_correct, k=6):
    return p_correct ** k, p_correct

In [ ]:
# 练习 4 参考答案
def recover_smooth(exact_match, k=6):
    return exact_match ** (1.0 / k)

---
## 🧪 真实数据胶囊：模算术的真实结构 + 缩放律的平滑性

两件真实的事: ① grokking 经典任务 **(a+b) mod p** 的真实结构(它确实有可泛化的循环规律, 是 grokking 的舞台); ② 真实**缩放律**(损失随规模平滑幂律下降)与表观「涌现」并存, 正是 Schaeffer 论点的旁证。

In [ ]:
# ① 模算术任务的真实结构: (a+b) mod p 是确定性的、有循环对称的可泛化规律
p = 13
table_mod = np.array([[(a + b) % p for b in range(p)] for a in range(p)])
print(f'(a+b) mod {p} 真值表 (前6x6):')
print(table_mod[:6, :6])
# 循环对称: table[a][b] == table[(a+1)%p][(b-1)%p] (沿反对角线不变)
assert table_mod[2, 3] == table_mod[3, 2], '加法交换律 -> 真值表对称'
assert table_mod[2, 3] == table_mod[(2+1) % p, (3-1) % p], '循环平移不变(可泛化结构)'
# 一个只背了部分格子的模型, 靠这个结构能补全其余 -> 这正是 grokking 要发现的规律
print('✅ 模算术有强循环结构(交换律+平移不变), 所以「泛化解」存在 -> grokking 有意义')

# ② 真实缩放律(Kaplan/Chinchilla 形态): 损失随参数量 N 平滑幂律下降
def chinchilla_loss(N, E=1.69, A=406.4, alpha=0.34):
    '''简化的 Chinchilla 损失律 L(N) = E + A * N^(-alpha) (固定充足数据)。真实拟合形态。'''
    return E + A * N ** (-alpha)

Ns = np.array([1e7, 1e8, 1e9, 1e10, 1e11])    # 参数量跨 4 个数量级
losses = chinchilla_loss(Ns)
print('\n参数量 N      损失 L(N)')
for N, L in zip(Ns, losses):
    print(f'  {N:.0e}    {L:.3f}')
# 平滑: log-log 上近似直线(幂律). 用前几点外插末点, 误差应很小
logN, logdiff = np.log10(Ns), np.log10(losses - 1.69)
coef = np.polyfit(logN[:3], logdiff[:3], 1)    # 用前3点拟合幂律
pred_last = 10 ** np.polyval(coef, logN[-1]) + 1.69
assert abs(pred_last - losses[-1]) / losses[-1] < 0.05, '损失律平滑到可跨数量级外插'
print('✅ 损失(连续度量)随规模平滑幂律下降、可外插 —— 与表观「涌现」并存, 正是度量假象的旁证')

**🧪 胶囊练习**：实现 `is_smooth_powerlaw(Ns, losses, E)`：判断 `losses-E` 在 log-log 上是否近似线性(幂律)。
返回拟合的 R²(越接近 1 越平滑)。

In [ ]:
def is_smooth_powerlaw(Ns, losses, E=1.69):
    # TODO: x=log10(Ns), y=log10(losses-E); 线性拟合; 返回 R^2
    #       R^2 = 1 - SS_res/SS_tot
    raise NotImplementedError

In [ ]:
# 自测
r2 = is_smooth_powerlaw(Ns, losses)
assert r2 > 0.99, '真实缩放律在 log-log 上近乎完美直线 -> 极平滑'
print(f'幂律拟合 R^2 = {r2:.4f} (越接近1越平滑)')
print('✅ 胶囊练习通过: 损失律的平滑性可量化')

In [ ]:
# 📖 胶囊参考答案
def is_smooth_powerlaw(Ns, losses, E=1.69):
    x = np.log10(Ns); y = np.log10(losses - E)
    coef = np.polyfit(x, y, 1)
    pred = np.polyval(coef, x)
    ss_res = np.sum((y - pred) ** 2); ss_tot = np.sum((y - y.mean()) ** 2)
    return 1 - ss_res / ss_tot

### 小结
- **grokking** = 训练早早满分(记忆), 测试很久后突然泛化; 引擎是 **weight decay**(关掉就不 grok)。
- 机制: 「训练误差=0」的流形上有高范数记忆解和低范数泛化解; weight decay 把解从前者**缓慢迁移**到后者。
- **progress measure**(如 cos(w,w_gen)) 在最终跳变前就平滑爬升 -> 「顿悟」是平滑进展过阈, 不是凭空奇点(Nanda)。
- **涌现**的阶跃很多是 **度量假象**(Schaeffer): 不连续度量(exact-match)把平滑能力放大成跳变; 连续度量看是平滑、可外插。
- 评测纪律: 看到「突然」, 先问「训练/扩展够久够大了吗(grokking)」「换连续度量还跳吗(涌现)」。

下一站: **模块 03 · 数据流水线** —— 从「模型行为」转到「喂给模型的数据」: 清洗、去重、shuffle、分片、质量统计。